In [50]:
import pandas as pd
import os, dill
import numpy as np
import cvxpy as cp
import plotly.express as px

from ecoli.processes.metabolism_redux_classic import NetworkFlowModel, FlowResult

os.chdir(os.path.expanduser('~/projects/vEcoli'))

%load_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [51]:
def load_sim(
        folder_path:str,
):
    """ This function is meant to load an output of a simulation in timeseries form.
        Note: This is not designed for parquet output format.
    """
    # --- Load Sim ---
    output = np.load(folder_path + '0_output.npy',allow_pickle='TRUE').item()
    output = output['agents']['0']
    fba = output['listeners']['fba_results']
    bulk = pd.DataFrame(output['bulk'])
    f = open(folder_path + 'agent_steps.pkl', 'rb')
    agent = dill.load(f)
    f.close()

    metabolism = agent['ecoli-metabolism-redux-classic']

    return fba, bulk, metabolism, output

# Define helper functions for standalone FBA

In [52]:
def get_subset_S(S, met_of_interest):
    S_met = S.loc[met_of_interest, :]
    S_met = S_met.loc[:,~np.all(S_met == 0, axis=0)]
    return S_met, S_met.columns

def get_keys(dict, value):
    return [key for key in dict if np.any(np.isin(value, dict[key]))]

def test_NetworkFlowModel(objective_weights, metabolism, fba,
                          uptake_addition = set([]),
                          uptake_removal = set([]),
                          new_exchange_molecules = set([]),
                          add_metabolite = None,
                          add_reaction = None,
                          remove_reaction = None,
                          add_kinetic = None,
                          force_reaction = None,
                          add_homeostatic_demand = None,
                          solver_choice=cp.GLOP):

    # --- Update Exchanges ---
    # Note: Exchange molecules are the molecules that can be secreted
    #       Uptake molecules are the molecules that can be taken up.
    uptake = metabolism.allowed_exchange_uptake.copy()
    uptake = set(uptake)
    uptake = uptake | uptake_addition
    uptake = uptake - uptake_removal

    exchange_molecules = metabolism.exchange_molecules.copy()
    exchange_molecules = exchange_molecules | new_exchange_molecules

    # --- Get whole-cell model outputs for FBA ---
    reaction_names = metabolism.reaction_names.copy()
    metabolites = metabolism.metabolite_names.copy()

    kinetic_reaction_ids = metabolism.kinetic_constraint_reactions.copy()
    kinetic = pd.DataFrame(fba["target_kinetic_fluxes"],
                           columns=metabolism.kinetic_constraint_reactions).mean(axis=0).copy() # units in counts/s
    homeostatic = pd.DataFrame(fba["target_homeostatic_dmdt"],
                               columns=metabolism.homeostatic_metabolites).mean(axis=0).copy() # units in counts/s
    homeostatic_counts = pd.DataFrame(fba["homeostatic_metabolite_counts"],
                                     columns=metabolism.homeostatic_metabolites).mean(axis=0).copy() # units in counts
    maintenance = pd.DataFrame(fba["maintenance_target"][1:], columns=['maintenance_reaction']).mean(axis=0).copy()

    S_new = metabolism.stoichiometry.copy()

    # --- Add/remove reactions, metabolites, kinetic constraints, homeostatic demands ---
    if add_metabolite is not None: #
        for m in add_metabolite:
            if m not in metabolites:
                metabolites.append(m)
        # append rows of zeros to S_new of length add_metabolite
        S_new = np.concatenate((S_new, np.zeros((len(add_metabolite), S_new.shape[1]))), axis=0)

    if add_reaction is not None:
        # Note: if one wants to add a reaction, it should be in a dictionary of
        #       form {"reaction_name": {"metabolite1": stoich1, "metabolite2": stoich2, ...}}
        assert isinstance(add_reaction, dict)

        for r,s in add_reaction.items():
            assert isinstance(s, dict)
            if r not in reaction_names:
                reaction_names.append(r)
            # append columns of reaction stoich to S_new of length add_reaction
            new_reaction = np.zeros((S_new.shape[0], 1))
            for m, v in s.items():
                new_reaction[metabolites.index(m), 0] = v
            S_new = np.concatenate((S_new, new_reaction), axis=1)

    if add_kinetic is not None:
        # Note: if one wants to add kinetics, it should be in a dictionary of
        #       form {"reaction_name": kinetic_target}
        assert isinstance(add_kinetic, dict)

        for r, v in add_kinetic.items():
            if r not in kinetic_reaction_ids:
                kinetic_reaction_ids.append(r)
                kinetic[r] = v
            if r in kinetic_reaction_ids:
                kinetic[r] = v

    if remove_reaction is not None:
        for r in remove_reaction:
            r_idx = reaction_names.index(r)
            S_new = np.delete(S_new, r_idx, axis=1)
            reaction_names.remove(r)
            if r in kinetic_reaction_ids:
                kinetic_reaction_ids.remove(r)
                del kinetic[r]

    if force_reaction is not None:
        force_reaction_idx = np.array([reaction_names.index(r) for r in force_reaction])
    else:
        force_reaction_idx = force_reaction

    if add_homeostatic_demand is not None:
        # assert add_homeostatic_demand is a list
        assert isinstance(add_homeostatic_demand, list)

        for met in add_homeostatic_demand:
            # here I just arbitrarily set the homeostatic demand to be 100 counts/s,
            # and homeostatic metabolite count to be 1, but this can be changed as needed.
            homeostatic[met] = 100
            homeostatic_counts[met] = 1

    # --- Solve NetworkFlowModel ---
    model = NetworkFlowModel(
            stoich_arr=S_new,
            metabolites=metabolites,
            reactions=reaction_names,
            homeostatic_metabolites=metabolism.homeostatic_metabolites,
            kinetic_reactions=kinetic_reaction_ids)
    model.set_up_exchanges(exchanges=exchange_molecules, uptakes=uptake)
    solution: FlowResult = model.solve(
            # homeostatic_concs=homeostatic_counts,
            # homeostatic_dm_targets=np.array(list(dict(homeostatic).values())),
            homeostatic_targets=np.array(list(dict(homeostatic).values())),
            maintenance_target=maintenance,
            kinetic_targets=np.array(list(dict(kinetic).values())),
            # binary_kinetic_idx=binary_kinetic_idx,
            binary_kinetic_idx=None,
            # force_flow_idx=force_reaction_idx,
            objective_weights=objective_weights,
            upper_flux_bound= 1000000000, # increase to 10^9 because notebook runs FlowResult using Counts, WC runs using conc.
            solver=solver_choice) #SCS. ECOS, MOSEK
    return solution.objective, solution.velocities, reaction_names, S_new, metabolites, kinetic

# Load Sim and run standalone FBA

In [53]:
time_num = 600
date = '2026-02-24'
experiment_name = 'homeo_diversity_8.53E-3'
condition = 'basal'
experiment_type = 'objective_weight'

time = str(time_num)
entry = f'{experiment_name}_{time}_{date}'
# folder_path = f'out/{experiment_type}/{condition}/{entry}/'
# ^^^ The above is just where I stored my sim output.

folder_path = "out/all_shrunk_row30_600_2026-03-19/"


fba, bulk, metabolism, output = load_sim(folder_path)

In [54]:
objective_weights = {'secretion': 0.00001, 'efficiency': 0.000001, 'kinetics': 0.00001, "homeostatic": 1}

# Simulate how flux would look like if we were to add a new reaction
# Note: This can be used if we want to simulate certain gene additions
add_reaction={
    'TEMP-REACTION-1': {
        'APO-CITRATE-LYASE[c]' : 1,
    },
    'TEMP-REACTION-2': {
        'CIT[c]':-1,
        'CITRATE-LYASE[c]' : -1,
    }
}

# Our model has a problem of not being able to partition flow.
# I often use the force_reaction argument to see if the model would still
# solve if we were to force some reaction to sustain flux.
# I use this argument to disentangle dead-end metabolites and reactions leading to them.
force_reaction = ['OXALODECARB-RXN']

# Simulate how the flux would look like if using citrate instead of glucose as the carbon source
uptake_addition = set(['CIT[p]'])
uptake_removal = set(['GLC[p]'])

# Simulate how flux would look like if we remove a downstream reaction of citrate degradation
# Note: This can be used if we want to simulate certain gene knockouts --
#       by removing the reactions that the gene/enzyme is involved in.
remove_reaction = ['CITTRANS-RXN']

In [55]:
result = test_NetworkFlowModel(objective_weights,
                               metabolism=metabolism,
                               fba=fba,
                               uptake_addition=uptake_addition,
                               force_reaction=force_reaction,
                               add_reaction=add_reaction,
                               remove_reaction=remove_reaction,
)

[oofv, solution_flux, test_reaction_names, S_new, metabolites_new, kinetics_new] = result

In [56]:
sim_flux = pd.DataFrame({
    'flux': solution_flux,
    'is_new': [
        'TEMP' if id in add_reaction.keys()
        else 'Original Reactions'
            for id in test_reaction_names
    ]
}, index=test_reaction_names)

This concludes running the stand-alone FBA. Now we can do various analysis on this output

In [57]:
# For example, we can look at the fluxes of reactions involving a certain metabolite of interest.
met_of_interest = ['CITRATE-LYASE[c]', 'OXALACETIC_ACID[c]', 'CIT[c]']
S_new = pd.DataFrame(S_new, index=metabolites_new, columns=test_reaction_names)

S_met, rxns  = get_subset_S(S_new, met_of_interest)

kinetic_reaction_ids = metabolism.kinetic_constraint_reactions.copy()
rxn_flux = sim_flux.loc[rxns]
rxn_flux['kinetic'] = [kinetics_new[r] if r in kinetic_reaction_ids else None for r in rxn_flux.index]
rxn_flux

,flux,is_new,kinetic
2-METHYLCITRATE-SYNTHASE-RXN,-0.000000e+00,Original Reactions,NaN
2.3.1.49-RXN,-0.000000e+00,Original Reactions,NaN
ACONITATEDEHYDR-RXN,9.877833e+01,Original Reactions,NaN
ACONITATEDEHYDR-RXN (reverse),-0.000000e+00,Original Reactions,NaN
ASPAMINOTRANS-RXN (reverse),1.845702e+08,Original Reactions,NaN
ASPAMINOTRANS-RXN__ASPAMINOTRANS-DIMER,7.184816e+05,Original Reactions,7.184816e+05
ASPAMINOTRANS-RXN__ASPAMINOTRANS-DIMER (reverse),2.154043e+06,Original Reactions,2.154043e+06
CITC-RXN,-0.000000e+00,Original Reactions,NaN
CITLY-RXN,-0.000000e+00,Original Reactions,NaN
CITRATE-PRO-3S-LYASE-THIOLESTERASE-RXN,-0.000000e+00,Original Reactions,NaN


#### Bar plot of fluxes of reactions involving the metabolite of interest

In [58]:
# Top 10 reactions by absolute flux
top10_flux = (
    sim_flux.assign(abs_flux=sim_flux["flux"].abs())
    .sort_values("abs_flux", ascending=False)
    .head(10)
    .copy()
)

# Keep labels in plotting order (largest first)
top10_flux = top10_flux.sort_values("abs_flux", ascending=True)

fig = px.bar(
    top10_flux,
    x="abs_flux",
    y=top10_flux.index,
    orientation="h",
    color="flux",  # preserves sign information
    color_continuous_scale="RdBu",
    labels={"abs_flux": "|Flux|", "y": "Reaction", "flux": "Signed flux"},
    title="Top 10 Reactions by Flux Magnitude",
)

fig.update_layout(
    yaxis_title="Reaction",
    xaxis_title="Absolute flux",
    template="plotly_white",
    height=500,
)
fig.update_xaxes(type="log")
fig.show()

### Scatterplot of the reactions with kinetic target

In [59]:
# Keep only reactions with a defined kinetic target
kinetic_compare = (
    rxn_flux[["flux", "kinetic"]]
    .dropna(subset=["kinetic"])
    .rename(columns={"flux": "simulated_flux", "kinetic": "target_kinetic_flux"})
    .copy()
)

# If there are no kinetic targets in this subset, this will print and skip plotting
if kinetic_compare.empty:
    print("No reactions in rxn_flux have kinetic targets.")
else:
    # Build symmetric range for y=x reference line
    min_val = min(
        kinetic_compare["simulated_flux"].min(),
        kinetic_compare["target_kinetic_flux"].min(),
    )
    max_val = max(
        kinetic_compare["simulated_flux"].max(),
        kinetic_compare["target_kinetic_flux"].max(),
    )

    fig = px.scatter(
        kinetic_compare.reset_index().rename(columns={"index": "reaction"}),
        x="target_kinetic_flux",
        y="simulated_flux",
        hover_data=["reaction"],
        title="Simulated vs Target Kinetic Flux (Reactions with Kinetic Targets)",
        labels={
            "target_kinetic_flux": "Target kinetic flux (count/s)",
            "simulated_flux": "Simulated flux (count/s)",
        },
        template="plotly_white",
    )

    # Add y = x line (perfect agreement)
    fig.add_shape(
        type="line",
        x0=min_val,
        y0=min_val,
        x1=max_val,
        y1=max_val,
        line=dict(color="gray", dash="dash"),
    )

    fig.update_layout(height=500)
    fig.update_xaxes(type="log")
    fig.update_yaxes(type="log")
    fig.show()